In [1]:
import os
import shutil
import sqlite3
import re

DB_FILE = "sec_filings_1000_1500.db"
BASE_DIR = "C:/Users/sudet/Desktop/to-be-processed"
PROCESSED_DIR = "C:/Users/sudet/Desktop/DONE-sec-edgar"

def get_existing_entries():
    """Fetch existing entries from the database."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("SELECT ticker, filing_type, date FROM filings")
    rows = cursor.fetchall()
    conn.close()
    return set(rows)

def extract_filing_date(file_path):
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
        match = re.search(r"FILED AS OF DATE:\s+(\d+)", content)
        return match.group(1) if match else None
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

def move_processed_files():
    existing_entries = get_existing_entries()
    moved = 0

    #print("Moving already-processed files...")
    for ticker in os.listdir(BASE_DIR):
        #print(f"\nProcessing {ticker}...")
        ticker_path = os.path.join(BASE_DIR, ticker)
        if not os.path.isdir(ticker_path):
            continue

        filing_type = "10-Q"
        #print(f"Checking {filing_type}...")
        filing_path = os.path.join(ticker_path, filing_type)
        if not os.path.exists(filing_path):
            continue

        for folder in os.listdir(filing_path):
            #print(f"Checking {folder}...")
            full_path = os.path.join(filing_path, folder, "full-submission.txt")
            if not os.path.exists(full_path):
                continue

            date = extract_filing_date(full_path)
            if not date:
                print(f"Could not extract date from {full_path}")
                continue

            key = (ticker, filing_type, date)
            if key in existing_entries:
                dest_path = os.path.join(PROCESSED_DIR, ticker, filing_type, folder)
                os.makedirs(dest_path, exist_ok=True)
                shutil.move(full_path, os.path.join(dest_path, "full-submission.txt"))
                moved += 1
                print(f"Moved {full_path} → {dest_path}")

    print(f"\n✅ Done. Moved {moved} already-processed files.")

# Run it
move_processed_files()

Moved C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0000950170-23-035168\full-submission.txt → C:/Users/sudet/Desktop/DONE-sec-edgar\AA\10-Q\0000950170-23-035168
Moved C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0000950170-23-055630\full-submission.txt → C:/Users/sudet/Desktop/DONE-sec-edgar\AA\10-Q\0000950170-23-055630
Moved C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0001564590-19-027529\full-submission.txt → C:/Users/sudet/Desktop/DONE-sec-edgar\AA\10-Q\0001564590-19-027529
Moved C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0001564590-19-039149\full-submission.txt → C:/Users/sudet/Desktop/DONE-sec-edgar\AA\10-Q\0001564590-19-039149
Moved C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0001564590-20-049105\full-submission.txt → C:/Users/sudet/Desktop/DONE-sec-edgar\AA\10-Q\0001564590-20-049105
Moved C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0001564590-21-039147\full-submission.txt → C:/Users/sudet/Desktop/DONE-sec-edgar\AA\10-Q\0001564590-21-039147
Moved C:/Users/sudet/D

KeyboardInterrupt: 

Use the below code to automatically delete empty file paths

In [1]:
import os
import shutil

BASE_DIR = "C:/Users/sudet/Desktop/to-be-processed"
DELETED_LOG = []

def is_empty_filing_folder(folder_path):
    """
    Check if a filing folder is empty or contains only an empty full-submission file.
    """
    full_txt = os.path.join(folder_path, "full-submission.txt")
    full_no_ext = os.path.join(folder_path, "full-submission")

    # If neither file exists, it's empty
    if not os.path.exists(full_txt) and not os.path.exists(full_no_ext):
        return True

    # If the file exists but is 0 bytes
    if os.path.exists(full_txt) and os.path.getsize(full_txt) == 0:
        return True
    if os.path.exists(full_no_ext) and os.path.getsize(full_no_ext) == 0:
        return True

    return False

def delete_empty_folders():
    deleted_count = 0

    for ticker in os.listdir(BASE_DIR):
        ticker_path = os.path.join(BASE_DIR, ticker)
        if not os.path.isdir(ticker_path):
            continue

        for form_type in os.listdir(ticker_path):
            form_path = os.path.join(ticker_path, form_type)
            if not os.path.isdir(form_path):
                continue

            for folder in os.listdir(form_path):
                filing_folder_path = os.path.join(form_path, folder)

                if not os.path.isdir(filing_folder_path):
                    continue

                if is_empty_filing_folder(filing_folder_path):
                    shutil.rmtree(filing_folder_path)
                    DELETED_LOG.append(filing_folder_path)
                    deleted_count += 1
                    print(f"🗑️ Deleted empty folder: {filing_folder_path}")

    print(f"\n✅ Done. Deleted {deleted_count} empty folders.")

# Run it
delete_empty_folders()

🗑️ Deleted empty folder: C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0000950170-23-035168
🗑️ Deleted empty folder: C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0000950170-23-055630
🗑️ Deleted empty folder: C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0001564590-19-027529
🗑️ Deleted empty folder: C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0001564590-19-039149
🗑️ Deleted empty folder: C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0001564590-20-049105
🗑️ Deleted empty folder: C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0001564590-21-039147
🗑️ Deleted empty folder: C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0001564590-21-052672
🗑️ Deleted empty folder: C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0001564590-22-026349
🗑️ Deleted empty folder: C:/Users/sudet/Desktop/to-be-processed\AA\10-Q\0001564590-22-035456

✅ Done. Deleted 9 empty folders.
